In [2]:
import torch

In [3]:
if not torch.cuda.is_available():
  raise RuntimeError(
      "No GPU is active."
  )
  gpu_name = torch.cuda.get_device_name(0)
  print(f"GPU: {gpu_name}")

In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("PyTorch cannot see the GPU.")

CUDA available: True
GPU count: 1
GPU name: Tesla T4


In [5]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!~/.local/bin/uv --version

downloading uv 0.13.6 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/bin/bash: line 1: /root/.local/bin/uv: No such file or directory


In [6]:
!uv --version

uv 0.13.6 (x86_64-unknown-linux-gnu)


In [7]:
!uv pip install --system "sentence-transformers==6.2.0"

Using Python 3.13.16 environment at: /usr
Resolved 59 packages in 710ms
Prepared 1 package in 72ms
Uninstalled 1 package in 12ms
Installed 1 package in 10ms
 - sentence-transformers==5.7.0
 + sentence-transformers==6.2.0


In [8]:
import torch
import sentence_transformers

print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA Available =: ", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU: ", torch.cuda.get_device_name(0))

sentence-transformers: 6.2.0
CUDA Available =:  True
GPU:  Tesla T4


In [9]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
MODEL_REVISION = "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"

model = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device="cuda",
)

print("Vector size:", model.get_embedding_dimension())
print("Maximum tokens: ", model.max_seq_length)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector size: 384
Maximum tokens:  512


In [10]:
import numpy as np

sample_texts = [
    "The vulnerability affects versions before 2.4.",
    "The patch was released in version 2.4."
]

sample_vectors = model.encode(
    sample_texts,
    batch_size=2,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
)

sample_vectors = sample_vectors.astype("float32")

print("Shape: ", sample_vectors.shape)
print("Type:", sample_vectors.dtype)
print("Vector lengths: ", np.linalg.norm(sample_vectors, axis=1))

Shape:  (2, 384)
Type: float32
Vector lengths:  [1. 1.]


In [11]:
from pathlib import Path
from urllib.request import urlretrieve

source_dir = Path("/content/benchmark_source")
source_dir.mkdir(parents=True, exist_ok=True)

feed_url = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-2025.json.gz"
feed_path = source_dir / "nvdcve-2.0-2025.json.gz"

urlretrieve(feed_url, feed_path)

size_mb = feed_path.stat().st_size / (1024 * 1024)
print(f"Saved: {feed_path}")
print(f"Compressed size: {size_mb:.2f} MB")

Saved: /content/benchmark_source/nvdcve-2.0-2025.json.gz
Compressed size: 24.52 MB


In [12]:
import gzip
import json

with gzip.open(feed_path, "rt", encoding="utf-8") as feed_file:
    nvd_data = json.load(feed_file)

cve_records = nvd_data["vulnerabilities"]

print(f"records={len(cve_records)}")
print(f"first_cve_id={cve_records[0]['cve']['id']}")

records=45448
first_cve_id=CVE-2025-0168


In [13]:
import numpy as np

target_count = 10_000
selected_positions = np.linspace(
    0,
    len(cve_records) - 1,
    num=target_count,
    dtype=int,
)

benchmark_cves = [cve_records[position] for position in selected_positions]

print(f"selected_records={len(benchmark_cves)}")
print(f"first_id={benchmark_cves[0]['cve']['id']}")
print(f"last_id={benchmark_cves[-1]['cve']['id']}")

selected_records=10000
first_id=CVE-2025-0168
last_id=CVE-2025-9462


In [14]:
sample_cve = benchmark_cves[0]["cve"]
print(f"id={sample_cve['id']}")

for description in sample_cve["descriptions"]:
  if description["lang"] == "en":
    print(description["value"])
    break

id=CVE-2025-0168
A vulnerability classified as critical has been found in code-projects Job Recruitment 1.0. This affects an unknown part of the file /_parse/_feedback_system.php. The manipulation of the argument person leads to sql injection. It is possible to initiate the attack remotely. The exploit has been disclosed to the public and may be used.


In [15]:
benchmark_documents = []
missing_english_descriptions = 0

for record in benchmark_cves:
    cve = record["cve"]
    english_description = ""

    for description in cve.get("descriptions", []):
        if description.get("lang") == "en":
            english_description = description.get("value", "").strip()
            break

    if not english_description:
        missing_english_descriptions += 1
        continue

    cve_id = cve["id"]
    text = f"CVE ID: {cve_id}\nDescription: {english_description}"

    benchmark_documents.append({
        "document_id": cve_id,
        "filename": f"{cve_id}.json",
        "location": "description",
        "text": text,
    })

print(f"documents={len(benchmark_documents)}")
print(f"missing_english_descriptions={missing_english_descriptions}")

documents=10000
missing_english_descriptions=0


In [16]:
token_counts = []

for document in benchmark_documents:
    token_ids = model.tokenizer.encode(
        document["text"],
        add_special_tokens=False,
    )
    token_counts.append(len(token_ids))

print(f"largest_document_tokens={max(token_counts)}")
print(f"documents_over_380={sum(count > 380 for count in token_counts)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1148 > 512). Running this sequence through the model will result in indexing errors


largest_document_tokens=2255
documents_over_380=285


In [17]:
from google.colab import files
uploaded = files.upload()

Saving production_rag_src_v2.zip to production_rag_src_v2.zip


In [18]:
from pathlib import Path
from zipfile import ZipFile
import sys

project_dir = Path("/content/project_source_v2")
project_dir.mkdir(parents=True, exist_ok=True)

with ZipFile("/content/production_rag_src_v2.zip") as archive:
    for name in archive.namelist():
        normalized_name = name.replace("\\", "/")

        if "__pycache__" in normalized_name or normalized_name.endswith(".pyc"):
            continue

        target = project_dir / normalized_name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))

for module_name in list(sys.modules):
    if module_name == "production_rag" or module_name.startswith("production_rag."):
        del sys.modules[module_name]

sys.path.insert(0, str(project_dir / "src"))

from production_rag.chunking import chunk_blocks

print("chunker_imported")
print("ingestion_loaded=", "production_rag.ingestion" in sys.modules)

chunker_imported
ingestion_loaded= False


In [19]:
from types import SimpleNamespace

BENCHMARK_TENANT_ID = "benchmark-tenant"
BENCHMARK_USER_ID = "benchmark-user"
benchmark_chunks = []

for document in benchmark_documents:
    block = SimpleNamespace(
        text=document["text"],
        location=document["location"],
        block_type="paragraph",
    )

    document_chunks = chunk_blocks(
        blocks=[block],
        document_id=document["document_id"],
        tenant_id=BENCHMARK_TENANT_ID,
        user_id=BENCHMARK_USER_ID,
        tokenizer=model.tokenizer,
        max_tokens=380,
        overlap_tokens=50,
    )

    for chunk in document_chunks:
        benchmark_chunks.append({
            "chunk_id": chunk.chunk_id,
            "document_id": chunk.document_id,
            "position": chunk.position,
            "text": chunk.text,
            "chunker_version": chunk.chunker_version,
            "tenant_id": chunk.tenant_id,
            "user_id": chunk.user_id,
            "filename": document["filename"],
            "location": chunk.location,
            "content_hash": chunk.content_hash,
        })

print(f"chunks={len(benchmark_chunks)}")
print(f"first_chunk_version={benchmark_chunks[0]['chunker_version']}")

chunks=10792
first_chunk_version=recursive-v1-size-380-overlap-50


In [21]:
chunk_token_counts = [
    len(model.tokenizer.encode(chunk["text"], add_special_tokens=False))
    for chunk in benchmark_chunks
]

print("largest_chunk_tokens=", max(chunk_token_counts))
print("chunks_over_380=", sum(count > 380 for count in chunk_token_counts))
print("tenant_ids=", set(chunk["tenant_id"] for chunk in benchmark_chunks))
print("user_ids=", set(chunk["user_id"] for chunk in benchmark_chunks))
print("first_chunk_text=", benchmark_chunks[0]["text"][:160])


largest_chunk_tokens= 380
chunks_over_380= 0
tenant_ids= {'benchmark-tenant'}
user_ids= {'benchmark-user'}
first_chunk_text= CVE ID: CVE-2025-0168
Description: A vulnerability classified as critical has been found in code-projects Job Recruitment 1.0. This affects an unknown part of t


In [22]:
import numpy as np

chunk_texts = [chunk["text"] for chunk in benchmark_chunks]

vectors = model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

vectors = vectors.astype(np.float32)

print("shape=", vectors.shape)
print("dtype=", vectors.dtype)
print("first_vector_length=", np.linalg.norm(vectors[0]))

Batches:   0%|          | 0/169 [00:00<?, ?it/s]

shape= (10792, 384)
dtype= float32
first_vector_length= 1.0


In [24]:
import json
from pathlib import Path

output_dir = Path("/content/benchmark_output")
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "vector.npy", vectors, allow_pickle=False)

with (output_dir / "chunks.jsonl").open("w", encoding="utf-8") as file:
  for chunk in benchmark_chunks:
    file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

saved_rows = sum(
    1 for _ in (output_dir / "chunks.jsonl").open(encoding="utf-8")
)

print("vector_count=", len(vectors))
print("metadata_rows=", saved_rows)
print("counts_match=", len(vectors) == saved_rows)

vector_count= 10792
metadata_rows= 10792
counts_match= True


In [25]:
loaded_revision = getattr(
    model[0].auto_model.config,
    "_commit_hash",
    None,
)

print("loaded_model_revision=", loaded_revision)

loaded_model_revision= None


In [26]:
from huggingface_hub import try_to_load_from_cache
from pathlib import Path

weights_path = try_to_load_from_cache(
    repo_id="BAAI/bge-small-en-v1.5",
    filename="model.safetensors",
)

print("weights_path=", weights_path)
if isinstance(weights_path, str):
    print("cached_revision=", Path(weights_path).parent.name)

weights_path= None


In [27]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"
MODEL_REVISION = "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"

model = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device="cuda",
)

print("model=", MODEL_NAME)
print("revision=", MODEL_REVISION)
print("device=", model.device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model= BAAI/bge-small-en-v1.5
revision= 5c38ec7c405ec4b44b94cc5a9bb96e735b38267a
device= cuda:0


In [28]:
vectors = model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

vectors = vectors.astype(np.float32)

print("shape=", vectors.shape)
print("dtype=", vectors.dtype)
print("first_vector_length=", np.linalg.norm(vectors[0]))

Batches:   0%|          | 0/169 [00:00<?, ?it/s]

shape= (10792, 384)
dtype= float32
first_vector_length= 1.0


In [29]:
vector_path = output_dir / "vectors.npy"
np.save(vector_path, vectors, allow_pickle=False)

saved_vectors = np.load(vector_path, allow_pickle=False)

print("saved_shape=", saved_vectors.shape)
print("saved_dtype=", saved_vectors.dtype)
print("saved_first_vector_length=", np.linalg.norm(saved_vectors[0]))

saved_shape= (10792, 384)
saved_dtype= float32
saved_first_vector_length= 1.0


In [32]:
import hashlib

metadata_path = output_dir / "chunks.jsonl"

chunker_versions = {
    chunk["chunker_version"] for chunk in benchmark_chunks
}

assert len(chunker_versions) == 1
assert len(vectors) == len(benchmark_chunks) == saved_rows

vector_norms = np.linalg.norm(vectors, axis=1)
vectors_normalized = bool(np.allclose(vector_norms, 1.0, atol=1e-5))

assert vectors_normalized

manifest = {
    "embedding_model": MODEL_NAME,
    "embedding_model_revision": MODEL_REVISION,
    "vector_size": int(vectors.shape[1]),
    "vectors_normalized": vectors_normalized,
    "chunker_version": next(iter(chunker_versions)),
    "chunk_count": int(vectors.shape[0]),
    "files": {
        "vectors.npy": hashlib.sha256(vector_path.read_bytes()).hexdigest(),
        "chunks.jsonl": hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
    },
}

manifest_path = output_dir / "manifest_json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(manifest)

{'embedding_model': 'BAAI/bge-small-en-v1.5', 'embedding_model_revision': '5c38ec7c405ec4b44b94cc5a9bb96e735b38267a', 'vector_size': 384, 'vectors_normalized': True, 'chunker_version': 'recursive-v1-size-380-overlap-50', 'chunk_count': 10792, 'files': {'vectors.npy': 'faa1feba2aaf32dc002dafa7721747bb399f3873dd266ffe9c98172195bd1f43', 'chunks.jsonl': 'a2cdf0b05e63c64e89400cd2f61e1980c901df286e56917535391eb74cb2dbff'}}


In [33]:
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
loaded_vectors = np.load(vector_path, allow_pickle=False)

actual_vector_hash = hashlib.sha256(vector_path.read_bytes()).hexdigest()
actual_metadata_hash = hashlib.sha256(metadata_path.read_bytes()).hexdigest()

with metadata_path.open(encoding="utf-8") as file:
    metadata_rows = sum(1 for _ in file)

checksums_match = (
    actual_vector_hash == saved_manifest["files"]["vectors.npy"]
    and actual_metadata_hash == saved_manifest["files"]["chunks.jsonl"]
)
counts_match = (
    loaded_vectors.shape[0]
    == metadata_rows
    == saved_manifest["chunk_count"]
)

print("checksums_match=", checksums_match)
print("counts_match=", counts_match)
assert checksums_match and counts_match

checksums_match= True
counts_match= True


In [35]:
print("output_dir=", output_dir)
print("files_in_output_dir=", [path.name for path in output_dir.iterdir()])
print("manifest_path=", manifest_path)
print("manifest_exists=", manifest_path.exists())

output_dir= /content/benchmark_output
files_in_output_dir= ['manifest_json', 'vectors.npy', 'chunks.jsonl', 'vector.npy']
manifest_path= /content/benchmark_output/manifest_json
manifest_exists= True


In [36]:
artifact_zip = Path("/content/benchmark_artifacts.zip")

with ZipFile(artifact_zip, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(output_dir / "vectors.npy", arcname="vectors.npy")
    archive.write(output_dir / "chunks.jsonl", arcname="chunks.jsonl")
    archive.write(manifest_path, arcname="manifest.json")

with ZipFile(artifact_zip, "r") as archive:
    print("zip_contents=", archive.namelist())

files.download(str(artifact_zip))

zip_contents= ['vectors.npy', 'chunks.jsonl', 'manifest.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>